# 🌾 MODULE 3: HUẤN LUYỆN MÔ HÌNH PHÂN LOẠI CHẤT LƯỢNG HẠT LÚA (DENSENET121) - BẢN TINH CHỈNH TỐI ƯU
**Đề tài Nghiên cứu Khoa học**: Hệ thống thị giác máy tính và học sâu phục vụ ước lượng số lượng và đánh giá phẩm cấp hạt giống lúa.

### 🎯 Các cải tiến tinh chỉnh chuyên sâu (Fine-Tuning v2):
1. **Nâng cấp độ phân giải**: Chuẩn $224 	imes 224$ của DenseNet121.
2. **Tăng cường dữ liệu (Data Augmentation)**: Xoay tự do $360^\circ$, lật ngang/dọc, chỉnh sáng nhẹ.
3. **Tự động cân bằng trọng số lớp (Class Weights)**: Tránh thiên vị giữa hạt nguyên và hạt khuyết tật.
4. **Mở rộng phạm vi Fine-tuning**: Mở khóa **60 layers cuối cùng** của DenseNet121 (thay vì chỉ 30 layers) để học sâu hơn các đặc trưng vi mô của hạt lúa.
5. **Tối ưu tốc độ học (Learning Rate)**: Bắt đầu Giai đoạn 2 với `lr = 5e-5` kết hợp `patience=5` cho `ReduceLROnPlateau` để không bị tụt tốc độ học quá sớm.
6. **Kiểm soát EarlyStopping chuẩn xác**: Tăng `patience=12` để mô hình có đủ thời gian bứt phá vượt ngưỡng $88\% - 92\%+$.
7. **Đánh giá & Xuất báo cáo khoa học**: Xuất ma trận nhầm lẫn (Confusion Matrix) và bảng F1-Score trên tập Test độc lập.


In [ ]:
# Bước 1: Kết nối Google Drive & Cài đặt thư viện
from google.colab import drive
drive.mount('/content/drive')

!pip install tensorflow scikit-learn matplotlib seaborn numpy


In [ ]:
# Bước 2: Import thư viện & Cấu hình Tham số
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

# ─────────────────────────────────────────────────────────────────────────────
# CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ HUẤN LUYỆN
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH = "/content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/"

# Đường dẫn folder chứa các thư mục nhãn: hat_nguyen và hat_khuyet_tat
DATASET_DIR = os.path.join(BASE_PATH, "DETECTED_OBJECTS/DATA1/OUTPUT_CROPPED_GRAINS")

# Thư mục lưu kết quả model sau khi train
SAVE_MODEL_DIR = os.path.join(BASE_PATH, "RESULTS/CNN_DenseNet121_Trained")
os.makedirs(SAVE_MODEL_DIR, exist_ok=True)

# Siêu tham số
IMG_SIZE   = (224, 224)  # Kích thước chuẩn của DenseNet121
BATCH_SIZE = 16
SEED       = 42

print("=" * 60)
print(f"📁 Thư mục dữ liệu     : {DATASET_DIR}")
print(f"💾 Thư mục lưu kết quả : {SAVE_MODEL_DIR}")
print(f"⚙️ Kích thước ảnh      : {IMG_SIZE} | Batch Size: {BATCH_SIZE}")
print("=" * 60)


In [ ]:
# Bước 3: Load Dữ liệu & Chia tập Train (70%) / Validation (15%) / Test (15%)
# 1. Load tập Train (70%)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.3,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# 2. Load 30% còn lại để tách làm Validation (15%) và Test (15%)
val_test_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.3,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names
num_val_batches = tf.data.experimental.cardinality(val_test_ds) // 2

val_ds  = val_test_ds.take(num_val_batches)
test_ds = val_test_ds.skip(num_val_batches)

print("=" * 60)
print(f"🎯 Các lớp nhận diện: {class_names}")
print(f"📊 Số batches: Train={len(train_ds)} | Validation={len(val_ds)} | Test={len(test_ds)}")
print("=" * 60)

# Tự động tính Class Weights cân bằng tỷ lệ mẫu
train_labels = []
for _, labels in train_ds.unbatch():
    train_labels.append(np.argmax(labels.numpy()))
train_labels = np.array(train_labels)

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights_dict = dict(enumerate(weights))
print(f"⚖️ Trọng số phạt cân bằng lớp (Class Weights): {class_weights_dict}")

# Tối ưu hóa pipeline nạp dữ liệu vào GPU
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.cache().prefetch(buffer_size=AUTOTUNE)


In [ ]:
# Bước 4: Thiết kế Khối Augmentation & Kiến trúc Mô hình DenseNet121
# 1. Khối tăng cường dữ liệu hạt lúa
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.5), # Xoay góc tự do
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.2)
], name="data_augmentation")

# 2. Khởi tạo Backbone DenseNet121
base_model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)
base_model.trainable = False  # Giai đoạn 1: Đóng băng backbone

# 3. Ghép nối thành Model hoàn chỉnh
inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = data_augmentation(inputs)
x = layers.Lambda(preprocess_input, name="densenet_preprocess")(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)

model = models.Model(inputs, outputs)
model.summary()


In [ ]:
# Bước 5: GIAI ĐOẠN 1 — Huấn luyện Warmup (Classifier Head)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase1 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6)
]

print("🚀 Bắt đầu Giai đoạn 1: Huấn luyện Classifier Head (25 Epochs)...")
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    class_weight=class_weights_dict,
    callbacks=callbacks_phase1
)


In [ ]:
# Bước 6: GIAI ĐOẠN 2 — Fine-Tuning 60 Layers Cuối Cùng của DenseNet121
# [CẢI TIẾN 1]: Mở khóa 60 layers cuối cùng để học sâu các đặc trưng vi mô của hạt lúa
base_model.trainable = True
for layer in base_model.layers[:-60]:
    layer.trainable = False

# [CẢI TIẾN 2]: Bắt đầu với Learning Rate tối ưu 5e-5 (thay vì 1e-5 quá nhỏ)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

best_model_path = os.path.join(SAVE_MODEL_DIR, "best_rice_densenet121.keras")

# [CẢI TIẾN 3]: Tăng patience của ReduceLR lên 5 và EarlyStopping lên 12
callbacks_phase2 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(best_model_path, monitor='val_accuracy', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)
]

print("🚀 Bắt đầu Giai đoạn 2: Fine-Tuning 60 layers sâu (50 Epochs)...")
history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    class_weight=class_weights_dict,
    callbacks=callbacks_phase2
)

# Lưu thêm 1 bản định dạng .h5 để tương thích ngược
legacy_model_path = os.path.join(SAVE_MODEL_DIR, "rice_grain_classifier_cnn.h5")
model.save(legacy_model_path)
print(f"🎉 Đã lưu model tối ưu tại: {best_model_path}")
print(f"📦 Đã lưu model tương thích ngược tại: {legacy_model_path}")


In [ ]:
# Bước 7: Vẽ Biểu đồ Loss & Accuracy qua 2 Giai đoạn Huấn luyện
acc  = history_phase1.history['accuracy'] + history_phase2.history['accuracy']
val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
loss = history_phase1.history['loss'] + history_phase2.history['loss']
val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(acc, label='Train Accuracy', color='#2196F3', lw=2)
plt.plot(val_acc, label='Validation Accuracy', color='#4CAF50', lw=2)
plt.axvline(x=len(history_phase1.history['accuracy'])-1, color='red', linestyle='--', label='Bắt đầu Fine-Tuning')
plt.title('Độ chính xác qua các Epochs (Accuracy)', fontweight='bold')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(loss, label='Train Loss', color='#FF9800', lw=2)
plt.plot(val_loss, label='Validation Loss', color='#F44336', lw=2)
plt.axvline(x=len(history_phase1.history['loss'])-1, color='red', linestyle='--', label='Bắt đầu Fine-Tuning')
plt.title('Độ hao phí qua các Epochs (Loss)', fontweight='bold')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

plt.tight_layout()
chart_save_path = os.path.join(SAVE_MODEL_DIR, "training_history.png")
plt.savefig(chart_save_path, dpi=300)
print(f"📊 Đã lưu biểu đồ huấn luyện tại: {chart_save_path}")
plt.show()


In [ ]:
# Bước 8: Đánh giá trên Tập Test Độc lập & Vẽ Ma trận Nhầm lẫn (Confusion Matrix)
print("🔍 Đang đánh giá mô hình trên tập Test độc lập...")
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

# 1. In Báo cáo phân loại chi tiết (Precision, Recall, F1-Score)
print("" + "=" * 60)
print("  BÁO CÁO PHÂN LOẠI CHI TIẾT TRÊN TẬP TEST (CLASSIFICATION REPORT)")
print("=" * 60)
report_text = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report_text)

# Lưu báo cáo vào file text
with open(os.path.join(SAVE_MODEL_DIR, "classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(report_text)

# 2. Vẽ Ma trận nhầm lẫn chuẩn hóa (Normalized Confusion Matrix)
cm = confusion_matrix(y_true, y_pred, normalize='true')
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='.2%', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title("Ma trận nhầm lẫn trên tập Test (Confusion Matrix)", fontweight='bold')
plt.xlabel("Nhãn dự đoán (Predicted)")
plt.ylabel("Nhãn thực tế (Ground Truth)")
plt.tight_layout()

cm_save_path = os.path.join(SAVE_MODEL_DIR, "test_confusion_matrix.png")
plt.savefig(cm_save_path, dpi=300)
print(f"🖼️ Đã lưu ma trận nhầm lẫn tại: {cm_save_path}")
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# 🎯 BƯỚC BỔ SUNG: DỰ ĐOÁN PHÂN LOẠI CHO ẢNH TÙY CHỌN (INFERENCE TEST)
# ─────────────────────────────────────────────────────────────────────────────
Sử dụng cell code bên dưới để upload ảnh lên `/content` hoặc chỉ định đường dẫn 1 ảnh hạt lúa bất kỳ để xem mô hình dự đoán nhãn (`hat_nguyen` hay `hat_khuyet_tat`) kèm biểu đồ xác suất phần trăm trực quan.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 🖼️ DỰ ĐOÁN NHÃN CHO 1 ẢNH HẠT LÚA BẤT KỲ
# ─────────────────────────────────────────────────────────────────────────────
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# [TÙY CHỌN 1]: Bật True nếu muốn hiện hộp thoại tải ảnh trực tiếp từ máy tính
UPLOAD_FROM_COMPUTER = False

# [TÙY CHỌN 2]: Hoặc nhập đường dẫn ảnh đã có sẵn trên Colab / Google Drive
CUSTOM_TEST_IMAGE_PATH = "/content/test_grain.png"

def predict_single_grain(image_path, trained_model, labels):
    if not os.path.exists(image_path):
        print(f"❌ Không tìm thấy file ảnh tại: {image_path}")
        print("💡 Gợi ý: Hãy upload ảnh lên Colab hoặc sửa lại biến CUSTOM_TEST_IMAGE_PATH.")
        return

    # 1. Đọc ảnh (hỗ trợ cả ảnh PNG trong suốt RGBA 4 kênh và ảnh màu 3 kênh)
    img_raw = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
    if img_raw is None:
        print(f"❌ Không thể đọc file ảnh: {image_path}")
        return

    # 2. Xử lý kênh Alpha (nếu ảnh PNG nền trong suốt từ SAHI)
    if len(img_raw.shape) == 3 and img_raw.shape[2] == 4:
        crop_bgr = img_raw[:, :, :3].copy()
        alpha = img_raw[:, :, 3]
        crop_bgr[alpha == 0] = [0, 0, 0]  # Đổi nền trong suốt thành màu đen
    else:
        crop_bgr = img_raw.copy()

    # 3. Tiền xử lý theo chuẩn kích thước mô hình (224x224)
    resized = cv2.resize(crop_bgr, (IMG_SIZE[0], IMG_SIZE[1]))
    rgb_img = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    input_tensor = np.expand_dims(rgb_img, axis=0)

    # 4. Dự đoán xác suất từ model
    preds = trained_model.predict(input_tensor, verbose=0)[0]
    pred_idx = int(np.argmax(preds))
    pred_label = labels[pred_idx]
    confidence = preds[pred_idx] * 100

    # 5. Hiển thị kết quả trực quan
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    display_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    axes[0].imshow(display_rgb)
    label_color = "#2E7D32" if pred_label == "hat_nguyen" else "#C62828"
    axes[0].set_title(f"Dự đoán: {pred_label.upper()}\nĐộ tự tin: {confidence:.2f}%", 
                      fontsize=13, fontweight="bold", color=label_color)
    axes[0].axis("off")

    y_pos = np.arange(len(labels))
    bar_colors = ["#4CAF50" if c == "hat_nguyen" else "#F44336" for c in labels]
    axes[1].barh(y_pos, preds * 100, color=bar_colors, alpha=0.85, edgecolor="black")
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels([c.upper() for c in labels], fontsize=11, fontweight="bold")
    axes[1].set_xlim(0, 100)
    axes[1].set_xlabel("Xác suất (%)", fontsize=11)
    axes[1].set_title("Phân phối xác suất 2 lớp", fontsize=12, fontweight="bold")
    axes[1].grid(axis="x", linestyle="--", alpha=0.5)

    for i, v in enumerate(preds * 100):
        axes[1].text(v + 1.5, i, f"{v:.2f}%", va="center", fontweight="bold")

    plt.tight_layout()
    plt.show()

    print("=" * 60)
    print(f"🌾 KẾT QUẢ DỰ ĐOÁN : {pred_label.upper()}")
    print(f"🎯 Độ tự tin       : {confidence:.2f}%")
    for c, p in zip(labels, preds):
        print(f"   - {c:15s}: {p*100:.2f}%")
    print("=" * 60)

# Thực thi
if UPLOAD_FROM_COMPUTER:
    print("📤 Hãy chọn 1 file ảnh hạt lúa từ máy tính để tải lên...")
    uploaded = files.upload()
    for fname in uploaded.keys():
        print(f"\n🚀 Đang phân tích ảnh: {fname}")
        predict_single_grain(fname, model, class_names)
else:
    predict_single_grain(CUSTOM_TEST_IMAGE_PATH, model, class_names)
